# Imports

In [8]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import ollama
import re

# Load data

In [2]:
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)


In [3]:
pdf_dir = "Статьи"
articles = []

for file in os.listdir(pdf_dir):
    if file.endswith(".pdf"):
        full_path = os.path.join(pdf_dir, file)
        text = extract_text_from_pdf(full_path)
        articles.append({
            "source": file,
            "text": text
        })

print("Статей загружено:", len(articles))

Статей загружено: 10


In [4]:
def split_into_paragraphs(text):
    text = re.sub(r'\n{2,}', '\n\n', text)

    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if len(p.strip()) > 50
    ]
    return paragraphs

In [5]:
def split_long_paragraph(paragraph, max_words=300):
    sentences = re.split(r'(?<=[.!?])\s+', paragraph)

    chunks = []
    current = []
    current_len = 0

    for s in sentences:
        s_len = len(s.split())

        if s_len > max_words:
            if current:
                chunks.append(" ".join(current))
                current = []
                current_len = 0
            chunks.append(s)
            continue

        if current_len + s_len > max_words:
            chunks.append(" ".join(current))
            current = []
            current_len = 0

        current.append(s)
        current_len += s_len

    if current:
        chunks.append(" ".join(current))

    return chunks

In [6]:
def chunk_text(
    text,
    max_words=300,
    overlap_paragraphs=1
):
    paragraphs = split_into_paragraphs(text)

    chunks = []
    current_chunk = []
    current_len = 0

    for p in paragraphs:
        p_len = len(p.split())

        if p_len > max_words:
            sub_paragraphs = split_long_paragraph(p, max_words)

            for sp in sub_paragraphs:
                sp_len = len(sp.split())

                if current_len + sp_len > max_words:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = []
                    current_len = 0

                current_chunk.append(sp)
                current_len += sp_len
            continue

        if current_len + p_len > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = (
                current_chunk[-overlap_paragraphs:]
                if overlap_paragraphs > 0 else []
            )
            current_len = sum(len(x.split()) for x in current_chunk)

        current_chunk.append(p)
        current_len += p_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [9]:
documents = []

for art in articles:
    chunks = chunk_text(
        art["text"],
        max_words=300,
        overlap_paragraphs=1
    )
    for ch in chunks:
        documents.append({
            "source": art["source"],
            "text": ch,
            "chunk_id": len(documents)
        })

print("Всего чанков:", len(documents))

Всего чанков: 341


In [10]:
model_emb = SentenceTransformer("all-MiniLM-L6-v2")
texts = [d["text"] for d in documents]

embeddings = model_emb.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [11]:
dim = embeddings.shape[1]
# index = faiss.IndexFlatL2(dim)
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Векторов в индексе:", index.ntotal)

Векторов в индексе: 341


# RAG

In [12]:
def analyze_question(question: str):
    q = question.lower()
    return {
        "fact": any(w in q for w in ["which", "who", "where", "institution", "acknowledg"]),
        "numeric": any(w in q for w in ["percentage", "variance", "sd", "mean", "stability"]),
        "results": any(w in q for w in ["result", "experiment", "stability", "variance"]),
    }

In [13]:
def retrieve(query, top_k=10, source_filter=None):
    """
    Поиск релевантных чанков по embedding.
    query: текст вопроса
    top_k: количество возвращаемых чанков
    source: опционально ограничить одной статьей (по имени файла)
    """
    q_emb = model_emb.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)

    distances, indices = index.search(q_emb, top_k)

    results = []
    for i in indices[0]:
        doc = documents[i]
        if source_filter is None or doc["source"] == source_filter:
            results.append(doc)

    return results

In [14]:
def retrieve_auto(question, source=None):
    """
    Автоматический RAG retrieval без фильтрации по секциям.
    question: текст вопроса
    source: ограничить одной статьей (опционально)
    """
    flags = analyze_question(question)
    top_k = 20 if (flags.get("fact") or flags.get("numeric")) else 10
    docs = retrieve(question, top_k=top_k, source_filter=source)


    return docs

In [15]:
BASE_PROMPT_GENERAL = """
Answer the question using ONLY the provided context from the scientific article.
Do NOT use any external knowledge or general background theory.
Report numerical values and specific terms exactly as stated.
If the answer is not present in the context, say so explicitly.
Be concise but complete, following the structure of the article.

Context:
{context}

Question:
{question}

Answer:
"""

In [24]:
BASE_PROMPT_TECH =  """
Answer the question using ONLY the provided context.

Rules:
- Do NOT use any external knowledge.
- Do NOT infer unstated motivations.
- Use only limitations, claims, and comparisons explicitly mentioned.
- If a limitation is not directly stated, do not invent it.
- Paraphrase faithfully without adding interpretation.

If the answer is not present in the context, say so explicitly.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
BASE_PROMPT_STATS = """
Answer the question using ONLY the provided context.

Rules:
- Do NOT use any external knowledge.
- Do NOT infer, explain, summarize, or interpret.
- Do NOT paraphrase numerical results.
- Report numerical values (SD, percentages, means) EXACTLY as stated.
- When asked to compare conditions, compare ONLY by the reported numerical values.
- Restrict the answer strictly to the experiment explicitly named in the question.
- If the information is not explicitly stated, say so.

Answer format:
- Highest stability (lowest SD): <condition> (Self-report SD = X; Observer SD = Y)
- Lowest stability (highest SD): <condition> (Self-report SD = X; Observer SD = Y)
- Largest variance contributor: <factor> (Self-report = X%; Observer = Y%)

Context:
{context}

Question:
{question}

Answer:
"""


In [16]:
BASE_PROMPT_MATH = """
You must answer by RESTATING the argument given in the text.

Rules:
- Use only objects, definitions, identities, and logical steps that explicitly appear in the provided context.
- Do NOT introduce intuition, interpretations, analogies, or external mathematical facts.
- Do NOT replace additive sum collisions with equalities of single terms.
- Preserve the structure of the proof: definitions → identities → conclusion.
- If an equality or construction is central to the proof, state it explicitly.
- If the text does not justify a claim, do not include it.

Context:
{context}

Question:
{question}

Answer:
"""

In [17]:
BASE_PROMPT_MATH_STRICT = """
You must answer by STRICTLY RESTATING what is written in the provided context.

Rules:
- Use ONLY terminology, notation, definitions, and constructions that explicitly appear in the context.
- Do NOT explain concepts, define standard terms, or add background knowledge.
- Do NOT interpret notation (e.g. do NOT explain what “π₀”, “exact”, or “Symp” mean).
- Do NOT paraphrase definitions: restate them in minimal syntactic form, preserving all symbols and equalities.
- If a group, object, or map is defined by an equality, quotient, or formula, you MUST state that expression explicitly.
- Do NOT infer unstated properties or meanings.
- If a term is not explicitly defined in the context, do NOT define or explain it.
- You are not allowed to introduce standard mathematical facts.
- If the answer is not explicitly stated in the context, reply EXACTLY:
  "The context does not state this."

Output structure (and nothing else):
1. Definitions exactly as stated.
2. Constructions or maps exactly as stated.

Context:
{context}

Question:
{question}

Answer:
"""

In [41]:
BASE_8 = """
Answer the question using ONLY information that is EXPLICITLY stated in the provided context.
Do NOT infer motivations, purposes, or implications unless they are directly stated.
Do NOT introduce new terms or concepts not present in the context.
If the answer is not explicitly stated, say so clearly.
Use the same terminology as the article.

If the question asks "why" or "what motivates" and the motivation is not explicitly stated,
answer by describing only the limitations or gaps that are explicitly mentioned,
or say clearly that the motivation is not explicitly stated in the context.

Be concise and factual. Avoid speculation or generalization.

Context:
{context}

Question:
{question}

Answer:
"""

In [35]:
def rag_answer(question, source=None, mode="general"):
    """
    Генерация ответа с использованием RAG.
    question: вопрос
    model_name: модель Ollama
    source: ограничение одной статьей
    mode: в зависимости от темы статьи
    """
    docs = retrieve_auto(question, source=source)
    
    if not docs:
        return "No relevant context found in the dataset."
        
    context = "\n\n".join([f"[{d['source']} | chunk {d['chunk_id']}]\n{d['text']}" for d in docs])

    if mode == "math":
        prompt = BASE_PROMPT_MATH.format(context=context, question=question)
        
    elif mode == "math_strict":
        prompt = BASE_PROMPT_MATH_STRICT.format(context=context, question=question)

    elif mode == "tech":
        prompt = BASE_PROMPT_TECH.format(context=context, question=question)

    elif mode == "stat":
        prompt = BASE_PROMPT_TECH.format(context=context, question=question)

    elif mode == "8":
        prompt = BASE_8.format(context=context, question=question)
        
    else:
        prompt = BASE_PROMPT_GENERAL.format(context=context, question=question)

    response = ollama.chat(
        model="qwen3:8b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]

In [19]:
def answer_without_rag(question, model_name="qwen3:8b"):
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "Answer concisely and factually."},
            {"role": "user", "content": question}
        ]
    )
    return response["message"]["content"]

# 7 Stable Personas: Dual-Assessment of Temporal Stability in LLM-Based Human Simulation

In [20]:
question_7_1 = (
    """
    According to Experiment I, which persona intensity condition (low, moderate, high) 
    shows the highest and lowest between-conversation stability based on the reported standard deviations,
    and which factor accounts for the largest proportion of variance in self-reported and observer-rated scores?"
    """
)

In [21]:
print("=== Без RAG ===")
print(answer_without_rag(question_7_1))

=== Без RAG ===
Based on Experiment I:  
- **Highest between-conversation stability** (lowest standard deviation) was observed in the **low intensity** condition.  
- **Lowest stability** (highest standard deviation) was in the **high intensity** condition.  
- The **factor accounting for the largest proportion of variance** in both self-reported and observer-rated scores was **persona intensity**.


In [22]:
print("\n=== С RAG ===")
print(rag_answer(question_7_1, source="7.pdf", mode="general"))


=== С RAG ===
**Answer:**

**Between-Conversation Stability:**  
- **Highest Stability:** *Moderate* intensity condition.  
  - Lower standard deviation (SD) values (e.g., 14.5 for "Moderate" vs. 20.3 for "High") indicate greater consistency across conversations.  
- **Lowest Stability:** *High* intensity condition.  
  - Higher SD values suggest greater variability in responses, implying less stability.  
  - *Low* intensity condition is not explicitly detailed in the provided data, but if inferred, it would likely show even lower SD than "Moderate."

**Factor Accounting for Largest Variance:**  
- **Model Consistency (SD):** The standard deviation of self-reported and observer-rated scores across conversations is the primary factor influencing variance. Lower SD (e.g., "Moderate") indicates higher stability, while higher SD (e.g., "High") reflects greater variability. This aligns with the observed patterns in the data, where model performance (e.g., consistency in responses) directl

In [23]:
print("\n=== С RAG ===")
print(rag_answer(question_7_1, source="7.pdf", mode="math"))


=== С RAG ===
**Answer:**

1. **Between-Conversation Stability (Based on Standard Deviations):**  
   - **Highest Stability (Lowest SD):** **Low Intensity**  
   - **Lowest Stability (Highest SD):** **High Intensity**  
   - **Reasoning:** The standard deviation (SD) values in the "Self-Report Observer" table (e.g., for "Moderate" intensity, self-report SD at turn 6 is 16.3, while for "High" intensity, it is 27.6) suggest that **high intensity** conditions exhibit greater variability in responses across conversations, indicating **lower stability**. Conversely, **low intensity** conditions show smaller SDs, implying **higher consistency** across conversations.

2. **Factor with the Largest Proportion of Variance:**  
   - **Persona Intensity (Low, Moderate, High)**  
   - **Reasoning:** The primary factor being tested in the experiment is the **intensity level** of the persona (low, moderate, high). Variability in self-reported and observer-rated scores is most likely explained by dif

In [27]:
print("\n=== С RAG ===")
print(rag_answer(question_7_1, source="7.pdf", mode="stat"))


=== С RAG ===
**Answer:**  
According to Experiment I, the **moderate** intensity condition shows the **highest between-conversation stability** (lowest standard deviations across turns), while the **high** intensity condition shows the **lowest stability** (highest standard deviations). This is inferred from the data tables, which indicate that Moderate intensity scores exhibit smaller variability (e.g., SD values of ~13.8–14.5) compared to High intensity (SD values of ~18.7–20.3).  

For the second part, the **model** (e.g., Claude, GPT, etc.) accounts for the **largest proportion of variance** in self-reported and observer-rated scores. This is supported by the significant differences in SD values across models, suggesting that the underlying language model's architecture and training data are the primary drivers of variability in responses.  

**Key Reasoning:**  
1. **Between-Conversation Stability:** Lower SD values (e.g., Moderate: ~13.8–14.5 vs. High: ~18.7–20.3) indicate high

# 8 Toward Non-Expert Customized Congestion Control

In [28]:
question_8_1 = (
    """
    What limitations of existing automated congestion control and networking code generation approaches
    motivate the authors to target non-expert users?
    """
)

In [29]:
print("=== Без RAG ===")
print(answer_without_rag(question_8_1))

=== Без RAG ===
Existing approaches often require deep expertise in networking protocols and coding, have steep learning curves, or generate complex, error-prone code. These limitations hinder non-expert users from effectively utilizing or modifying congestion control and networking systems, motivating the development of more accessible, user-friendly tools tailored for non-expert audiences.


In [33]:
print("\n=== С RAG ===")
print(rag_answer(question_8_1, source="8.pdf", mode = "general"))


=== С RAG ===
The authors identify several limitations in existing automated congestion control and networking code generation approaches that motivated their focus on non-expert users:  

1. **Inadequate Handling of Complex, Long-Code Tasks**:  
   Traditional code generation methods (e.g., direct LLM-based approaches) struggle with long, complex class-level coding tasks, leading to suboptimal results. This limits their ability to generate robust, customized congestion control algorithms (CCAs) that meet specific user requirements.  

2. **Lack of User-Centric Design for Non-Experts**:  
   Existing frameworks do not provide intuitive interfaces or workflows for non-expert users to specify their requirements (e.g., streaming resolution, bandwidth needs) or integrate home network conditions. This creates a barrier for users without technical expertise to configure or customize CCAs.  

3. **Insufficient Safety and Deployment Guarantees**:  
   Prior approaches often fail to enforce sa

In [42]:
question_8_2 = "What limitations of existing automated congestion control and networking code generation approaches are explicitly mentioned in the related work section?"



In [43]:
print("\n=== С RAG ===")
print(rag_answer(question_8_2, source="8.pdf", mode = "8"))


=== С RAG ===
The related work section explicitly highlights the following limitations of existing automated congestion control and networking code generation approaches:

1. **Focus on Rule-Level Learning vs. System-Level Code Generation**:  
   Existing automated CCA design approaches (e.g., learning-based methods) primarily focus on **rule-level learning** (e.g., policy-switching strategies or dynamic variable control). These methods do not address the generation of **system-level executable code** that can be directly deployed as Linux-compatible congestion control algorithms (CCAs). This contrasts with the proposed work, which emphasizes generating **executable code** for congestion control systems.

2. **Reliance on Third-Party Libraries**:  
   Some prior approaches (e.g., the work in [4]) synthesize objectives and propose heuristic CCA design methods but implement them using **third-party libraries**. This dependency on external tools may limit integration with native Linux ke

# 9 Minimal-Action Discrete Schrödinger Bridge Matching for Peptide Sequence Design

In [44]:
question_9_1 = (
    """
    How does MadSBM compare to the discrete diffusion baseline (EvoFlow)
    in terms of unconditional peptide sequence generation quality across different sampling budgets?
    """
)

In [45]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_1))

print("\n=== С RAG ===")
print(rag_answer(question_9_1, source="9.pdf", mode = "8"))

=== Без RAG ===
MadSBM outperforms the discrete diffusion baseline (EvoFlow) in unconditional peptide sequence generation quality across different sampling budgets. MadSBM achieves higher quality and diversity with fewer sampling steps due to its continuous score-based sampling mechanism, which enables more efficient exploration of the sequence space compared to EvoFlow's discrete diffusion process. This efficiency allows MadSBM to maintain superior performance even under constrained sampling budgets.

=== С RAG ===
MadSBM outperforms the discrete diffusion baseline (EvoFlow) in unconditional peptide sequence generation quality across different sampling budgets, particularly under low sampling constraints. Here's a structured comparison:

### **Key Advantages of MadSBM Over EvoFlow:**
1. **Improved Sample Efficiency**:  
   - MadSBM leverages a **simulation-free learning rule** based on a cross-entropy objective, eliminating the need for explicit forward–backward bridge solvers. This r

In [50]:
print("\n=== С RAG ===")
print(rag_answer(question_9_1, source="9.pdf", mode = "general"))


=== С RAG ===
MadSBM demonstrates superior performance compared to the discrete diffusion baseline (EvoFlow) in unconditional peptide sequence generation, particularly under **low sampling budgets**. Here's a structured comparison:

### **Key Advantages of MadSBM Over EvoFlow**
1. **Efficient Sampling Without Fixed Interpolation Schemes**  
   - EvoFlow, like traditional diffusion models, relies on fixed interpolation schemes (e.g., noise-to-data progression) to generate sequences. These methods often require extensive sampling steps to reach high-likelihood regions.  
   - MadSBM, by contrast, models sequence evolution as a **controlled continuous-time Markov chain** on the amino-acid edit graph. It uses a **learnable control field** to steer samples directly toward high-probability regions, **reducing the need for prolonged sampling**. This enables **higher quality generation with fewer samples**.

2. **Stability and Sample Efficiency**  
   - The theoretical framework of MadSBM (e.

In [62]:
question_9_2 = (
    """
    What are the observed pseudo-perplexity (PPL) and pLDDT scores
    for MadSBM and the discrete diffusion baseline (EvoFlow)
    in unconditional peptide sequence generation across different sampling budgets?
    """
)


In [63]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_2))

=== Без RAG ===
The observed pseudo-perplexity (PPL) and pLDDT scores for MadSBM and EvoFlow in unconditional peptide sequence generation vary with sampling budgets. For example, MadSBM typically achieves lower PPL (indicating better sequence quality) and higher pLDDT (better structural accuracy) than EvoFlow, especially with larger sampling budgets. However, exact values depend on experimental setups and are detailed in specific benchmarking studies. Refer to the original research for precise metrics.


In [64]:
print("\n=== С RAG ===")
print(rag_answer(question_9_2, source="9.pdf", mode = "general"))


=== С RAG ===
The provided text does not explicitly report observed pseudo-perplexity (PPL) or pLDDT scores for MadSBM and the discrete diffusion baseline (EvoFlow) in unconditional peptide sequence generation across different sampling budgets. However, it does describe the relative performance of MadSBM compared to such baselines:

1. **Pseudo-Perplexity (PPL):**  
   - MadSBM is evaluated using the PPL metric, which measures biological plausibility by masking tokens and computing the average negative log-likelihood (NLL) via the ESM-2 language modeling head. The text highlights that MadSBM achieves **improved perplexity** at **substantially lower sampling budgets** compared to discrete diffusion and flow-based baselines (e.g., EvoFlow). This suggests that MadSBM generates more biologically plausible sequences with fewer sampling steps.

2. **Structural Confidence (pLDDT):**  
   - While pLDDT scores (used in protein structure prediction) are not explicitly mentioned in the text, it 

In [65]:
print("\n=== С RAG ===")
print(rag_answer(question_9_2, source="9.pdf", mode = "math"))


=== С RAG ===
The provided text discusses the **pseudo-perplexity (PPL)** metric for MadSBM in the context of unconditional peptide sequence generation. PPL is calculated using ESM-2's masked language modeling head, where one token is masked at a time, and the negative log-likelihood (NLL) of the resulting sequence is averaged across all positions. This serves as a measure of biological plausibility. However, **pLDDT scores** (a structural confidence metric, often used in AlphaFold for predicting ligand-docking distances) are **not explicitly mentioned** in the text. 

### Summary:
- **PPL for MadSBM**: The text describes PPL as a metric derived from ESM-2, but specific numerical values or comparisons to a discrete diffusion baseline (e.g., EvoFlow) are **not provided** in the given content.
- **pLDDT Scores**: The text does **not mention** pLDDT scores at all, so no data is available for this metric.

### Conclusion:
The text provides theoretical and methodological details about PPL 

In [71]:
question_9_3 = ("""How is generative sampling performed after training, and how are transition rates, jump probabilities, and token updates defined in the CTMC-based sampling procedure?""")

In [72]:
question_9_3

'How is generative sampling performed after training, and how are transition rates, jump probabilities, and token updates defined in the CTMC-based sampling procedure?'

In [73]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_3))

=== Без RAG ===
Generative sampling in a Continuous-Time Markov Chain (CTMC) framework after training involves simulating the CTMC's transitions to generate sequences. Transition rates are defined by the Q-matrix, where $ Q_{ij} $ (for $ i \neq j $) represents the rate of transitioning from state $ i $ to state $ j $, and $ Q_{ii} = -\sum_{j \neq i} Q_{ij} $. Jump probabilities are derived by normalizing the transition rates: $ P_{ij} = \frac{Q_{ij}}{-Q_{ii}} $ for $ i \neq j $. Token updates occur by selecting the next state based on these probabilities, with the time between transitions following an exponential distribution with rate $ -Q_{ii} $. This process is often implemented using algorithms like the Gillespie algorithm to simulate the CTMC dynamics.


In [74]:
print("\n=== С RAG ===")
print(rag_answer(question_9_3, source="9.pdf", mode = "math"))


=== С RAG ===
Generative sampling in MadSBM is performed using a **controlled continuous-time Markov chain (CTMC)** framework, where the learned control field $ u_\theta $ modifies the reference transition rates $ R_0 $ to steer the process toward the data distribution. Here's how the key components are defined and executed:

---

### **1. Transition Rates and Control Field**
- **Reference Process $ R_0 $**:  
  The base transition rates are derived from a pre-trained encoder-only protein language model (e.g., ESM-2). Logits $ f_\phi(x_t) \in \mathbb{R}^{L \times V} $ from the frozen ESM-2 model are used as reference scores, encoding token plausibility for amino-acid sequences.  
  $$
  \log R_0(x, x') = f_\phi(x) \quad \text{(local plausibility)}
  $$

- **Control Field $ u_\theta $**:  
  The learned control field tilts the reference rates to guide the process toward the data distribution. Transition rates decompose as:  
  $$
  \log R_u(x, x') = \log R_0(x, x') + u_\theta(x, x', t)

# 10 Endogenous Inequality Aversion: Decision criteria for triage and other ethical tradeoffs

In [75]:
question_10_1 = "How do the mixing and homotheticity axioms in the model lead to the representation of social welfare through a “fan” function with weights endogenously dependent on the level of welfare?"


In [76]:
print("=== Без RAG ===")
print(answer_without_rag(question_10_1))

=== Без RAG ===
The mixing and homotheticity axioms ensure that the social welfare function is a convex combination of individual utilities (mixing) and scales proportionally with welfare levels (homotheticity). These axioms lead to a "fan" function, where social welfare is represented as a weighted sum of utilities, with weights endogenously determined by the welfare level. Homotheticity enforces that weights adjust proportionally to maintain consistency across different welfare states, while mixing ensures the aggregation reflects individual preferences. This structure allows weights to vary endogenously, capturing dynamic societal priorities.


In [54]:
print("\n=== С RAG ===") # 3 лучший самый
print(rag_answer(question_10_1, source="10.pdf", mode = "tech"))


=== С RAG ===
The mixing and homotheticity axioms in the model lead to the representation of social welfare through a "fan" function with endogenously dependent weights by enforcing consistency and adaptability in how welfare is evaluated across different contexts. Here's a structured breakdown:

---

### **1. Mixing Invariance Axiom**
- **Purpose**: Ensures that the social welfare function remains invariant under convex combinations of allocations. That is, if two allocations are equally valid, the welfare function should not favor one over the other based on arbitrary mixing proportions.
- **Implication**: 
  - Forces the weights to depend **only** on the **current welfare level** (not on arbitrary allocation mixtures). 
  - Eliminates exogenous weight specifications, ensuring the function is self-referential: the weights are determined by the system's state (e.g., aggregate welfare), not by external criteria.

---

### **2. Homotheticity Axiom**
- **Purpose**: Requires that the soc

In [58]:
print("\n=== С RAG ===") # 7 лучший
print(rag_answer(question_10_1, mode = "stat"))


=== С RAG ===
The mixing and homotheticity axioms in the model lead to the representation of social welfare through a "fan" function with endogenously determined weights by establishing structural consistency and self-referentiality. Here's how:

### 1. **Mixing Invariance (Consistency under Probabilistic Combinations)**  
   - **Role**: Mixing invariance ensures that the social welfare function is **linear in probabilities**. That is, the preference between two allocations remains unchanged when they are combined with arbitrary probabilities.  
   - **Implication**: This axiom rules out non-linear or context-dependent aggregation rules, forcing the welfare function to be **additively separable** in terms of individual utilities. It ensures that the weights (e.g., Rawlsian or utilitarian) assigned to agents are **consistent** across different allocations, even when probabilities are involved.  
   - **Connection to the Fan Function**: By requiring linearity under mixing, the model ens